# tp-16b 特徴量 bake-off（2 回目）: 後日談の変種とクッション値・含水率（訂正後データ）

tp-16 で最も有望だった**後日談（AF）**の窓・減衰・指標の変種と、**入れ替わりを訂正したクッション値・含水率（CU）**を再評価する。基準・採否・評価は tp-16 と同じ（valid 2022〜2024 × 2 seed、勝率の差の 95% 上限 &lt; 0・効果 ≥ 0.0002・連対/複勝が 0.0002 を超えて悪化しない。test 2025 は使わない）。

| 群 | 内容 |
|----|------|
| AF_l2 / AF_l3 / AF_l5 | 直近 2 / 3 / 5 走の相手の次走成績（勝率・複勝率・着順の残差。分かる分だけ） |
| AF_l5_d07 | 5 走、1 走ずつ重みを 0.7 倍に減衰 |
| CU_fixed | クッション値・含水率（8 列。4 コーナー含水率の入れ替わりを訂正済みのデータ） |
| CU_goal | 上のうち 4 コーナー系（cu_moist_4c, cu_moist_diff）を除く 6 列 |

In [ ]:
import os, sys
from pathlib import Path


def _find_root():
    p = Path.cwd().resolve()
    for q in [p, *p.parents]:
        if (q / "AGENTS.md").exists() and (q / "src").is_dir():
            return q
    raise RuntimeError("リポジトリルートが見つかりません")


REPO_ROOT = _find_root()
os.chdir(REPO_ROOT)
os.environ.setdefault("POLARS_MAX_THREADS", "4")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from src.utils.project_env import load_project_dotenv
load_project_dotenv()

In [ ]:
import json, subprocess, gc, time
import numpy as np
import pandas as pd
import mlflow

from src.pipeline.mlflow.train_utils import require_tracking_server
from src.pipeline.models import tp05_data, zoo, zoo_runner, feature_bakeoff as fb
from src.pipeline.models.compute import detect_compute
from src.pipeline.features import bakeoff as bo

WORK = REPO_ROOT / "data/local/modeling/training_pipeline"
OUT = WORK / "bakeoff_b"
OUT.mkdir(parents=True, exist_ok=True)
THREADS = int(os.environ.get("TP16B_THREADS", detect_compute(reserve_cores=6).threads))
SEEDS = [0, 1]
SMOKE = os.environ.get("TP16B_SMOKE") == "1"
mlflow.set_experiment("keiba-tp-feature-bakeoff")
require_tracking_server()
GIT_COMMIT = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True, cwd=REPO_ROOT).stdout.strip()
D = tp05_data.load_dataset(WORK)
df, meta, folds = D["df"], D["meta"], D["folds"]
EP, EXTRAS, PED = tp05_data.load_extras_ped(df, WORK)
X0 = pd.concat([df[D["FEATS"]["base+S+C1"]], EP], axis=1)
best = json.loads((WORK / "hpo/best_params.json").read_text(encoding="utf-8"))["best"]
LP = zoo.tuned_params("lgb", best["lgb"]["params"])
if SMOKE:
    folds = folds[:1]
print("行", len(df), "| 基準の列数", X0.shape[1], "| スレッド", THREADS, "| smoke", SMOKE)

## 候補の特徴量を作る（すべてそのレースの日より前に分かる情報）

In [ ]:
base_cols = ["race_id", "horse_id", "date", "venue_code", "surface", "distance", "finish_position", "field_size"]
dd = df[base_cols].copy()
G = {}
def build(name, fn):
    t0 = time.time()
    f = fn()
    G[name] = f
    f.assign(race_id=df["race_id"].to_numpy(), horse_id=df["horse_id"].to_numpy()).to_parquet(OUT / f"cand_{name}.parquet", index=False)
    print(f"{name}: {f.shape[1]} 列, 欠損率 {f.isna().mean().mean():.3f}, {time.time() - t0:.0f}s", flush=True)

build("AF_l2", lambda: bo.aftermath_features(dd, lags=2))
build("AF_l3", lambda: bo.aftermath_features(dd, lags=3))
build("AF_l5", lambda: bo.aftermath_features(dd, lags=5))
build("AF_l5_d07", lambda: bo.aftermath_features(dd, lags=5, decay=0.7))
build("CU_fixed", lambda: bo.cushion_features(dd, bo.load_cushion(REPO_ROOT / "data/jra_baba")))
G["CU_goal"] = G["CU_fixed"].drop(columns=["cu_moist_4c", "cu_moist_diff"])
display(pd.DataFrame({k: [v.shape[1], float(v.isna().mean().mean())] for k, v in G.items()}, index=["列数", "平均欠損率"]).T.round(3))

## 基準と各群の比較（LightGBM、2 seed × 3 fold）

In [ ]:
key = ["race_id", "horse_id"]
fin_map = meta.set_index(key)["finish_position"]
res0, OOF0 = zoo_runner.run_oof("lgb_bake_base", "lgb_binary", X0, meta, folds, SEEDS, THREADS, params=dict(LP))
print(f"基準: win {res0['logloss_win'].mean():.5f} top2 {res0['logloss_top2'].mean():.5f} top3 {res0['logloss_top3'].mean():.5f}", flush=True)
ROWS = []
names = list(G) if not SMOKE else ["AF_l2", "CU_goal"]
for name in names:
    t0 = time.time()
    X = pd.concat([X0, G[name].add_prefix(f"bk_{name}_")], axis=1)
    res, oof = zoo_runner.run_oof(f"lgb_bake_{name}", "lgb_binary", X, meta, folds, SEEDS, THREADS, params=dict(LP))
    ROWS.append(fb.compare_oof(OOF0, oof, fin_map, name))
    print(f"{name}: win {res['logloss_win'].mean():.5f} Δ {ROWS[-1]['Δwin']:+.5f} (上限 {ROWS[-1]['win_hi']:+.5f}) top2 {ROWS[-1]['Δtop2']:+.5f} top3 {ROWS[-1]['Δtop3']:+.5f} 採用={ROWS[-1]['採用']} ({time.time() - t0:.0f}s)", flush=True)
    pd.DataFrame(ROWS).set_index("variant").to_csv(OUT / "bakeoff_vs_base.csv")
    gc.collect()
R = pd.DataFrame(ROWS).set_index("variant")
display(R[["win_base", "win_new", "Δwin", "win_hi", "Δtop2", "Δtop3", "採用"]].round(5))

## 採用した群をまとめて足した場合（相互に重複しないか）

In [ ]:
adopted = [n for n in R.index if bool(R.loc[n, "採用"])]
print("単独で採用基準を満たした群:", adopted)
COMB = None
if len(adopted) >= 2:
    Xc = pd.concat([X0] + [G[n].add_prefix(f"bk_{n}_") for n in adopted], axis=1)
    resc, oofc = zoo_runner.run_oof("lgb_bake_combined", "lgb_binary", Xc, meta, folds, SEEDS, THREADS, params=dict(LP))
    COMB = fb.compare_oof(OOF0, oofc, fin_map, "+".join(adopted))
    print(f"まとめて: win {resc['logloss_win'].mean():.5f} Δ {COMB['Δwin']:+.5f} (上限 {COMB['win_hi']:+.5f}) top2 {COMB['Δtop2']:+.5f} top3 {COMB['Δtop3']:+.5f}")

In [ ]:
# 寄与（gain）と、既存特徴量との関係
import lightgbm as lgb
y = (meta["finish_position"].to_numpy() == 1).astype(int)
top = [n for n in R.sort_values("Δwin").index[:3]]
f = folds[-1]
gain_rows = {}
for n in top:
    Xb = pd.concat([X0, G[n].add_prefix(f"bk_{n}_")], axis=1)
    cats = [c for c in Xb.columns if str(Xb[c].dtype) == "category"]
    prm = dict({"objective": "binary", "verbosity": -1, "num_threads": THREADS, "seed": 0, "bagging_freq": 1}, **LP["lgb"])
    bst = lgb.train(prm, lgb.Dataset(Xb.iloc[f.train_index], label=y[f.train_index], categorical_feature=cats), num_boost_round=LP["n_rounds"])
    g = pd.Series(bst.feature_importance("gain"), index=Xb.columns); g = g / g.sum()
    new = [c for c in Xb.columns if c.startswith("bk_")]
    gain_rows[n] = float(g[new].sum()) * 100
    print(n, "gain 合計 %.2f%%" % gain_rows[n]); display(g[new].sort_values(ascending=False).head(6).round(4))
report = {"git_commit": GIT_COMMIT, "results": R.round(6).reset_index().to_dict("records"), "adopted": adopted, "combined": COMB, "gain_share_pct": gain_rows}
(OUT / "report.json").write_text(json.dumps(report, ensure_ascii=False, indent=1, default=str), encoding="utf-8")
idx = {f"g{i}": n for i, n in enumerate(R.index)}
(OUT / "metric_index.json").write_text(json.dumps(idx, ensure_ascii=False), encoding="utf-8")
with mlflow.start_run(run_name="tp-16b-feature-bakeoff", tags={"notebook": "tp-16b", "git_commit": GIT_COMMIT}):
    for i, n_ in enumerate(R.index):
        mlflow.log_metrics({f"delta_win_g{i}": float(R.loc[n_, "Δwin"])})
    for f_ in ("bakeoff_vs_base.csv", "report.json", "metric_index.json"):
        mlflow.log_artifact(str(OUT / f_), "bakeoff")